# NumPy vectorization walkthrough

Shapes, broadcasting, and why loops are the wrong tool for array math. Everything here uses the tested functions in `datapipe.numpy_tools`. All data is randomly generated with a fixed seed.

In [1]:
import sys, time
sys.path.insert(0, '../src')
import numpy as np
from datapipe.numpy_tools import zscore, pairwise_euclidean, cosine_similarity, top_k_indices, softmax

rng = np.random.default_rng(7)
print('numpy', np.__version__)

numpy 2.5.3


## 1. Broadcasting: the shape rule

NumPy aligns shapes from the right. Dimensions match if they are equal or one of them is 1; the size-1 dimension is stretched.

In [2]:
a = np.arange(3).reshape(3, 1)   # (3, 1)
b = np.arange(4).reshape(1, 4)   # (1, 4)
print((a + b).shape)
print(a + b)

(3, 4)
[[0 1 2 3]
 [1 2 3 4]
 [2 3 4 5]]


## 2. Standardizing columns without a loop

Each column gets mean 0 and standard deviation 1 in one expression.

In [3]:
X = rng.normal(loc=[10, 50, 200], scale=[1, 5, 40], size=(1000, 3))
Z = zscore(X)
print('before  mean:', X.mean(axis=0).round(2), ' std:', X.std(axis=0).round(2))
print('after   mean:', Z.mean(axis=0).round(2), ' std:', Z.std(axis=0).round(2))

before  mean: [  9.95  49.77 200.03]  std: [ 1.    4.93 39.53]
after   mean: [0. 0. 0.]  std: [1. 1. 1.]


## 3. Loop vs vectorized: pairwise distances

500 points in 8 dimensions, every point against every other.

In [4]:
P = rng.normal(size=(500, 8))

def loop_dist(P):
    n = len(P)
    out = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((P[i] - P[j]) ** 2).sum())
    return out

t = time.perf_counter(); D_loop = loop_dist(P); t_loop = time.perf_counter() - t
t = time.perf_counter(); D_vec = pairwise_euclidean(P); t_vec = time.perf_counter() - t
print('same result:', np.allclose(D_loop, D_vec))
print(f'loop: {t_loop:.3f}s   vectorized: {t_vec:.4f}s   speedup: {t_loop / t_vec:.0f}x')

same result: True
loop: 0.582s   vectorized: 0.0256s   speedup: 23x


## 4. A tiny retrieval example

Cosine similarity plus top-k is the core of embedding search. Here the vectors are random, so this shows the mechanics, not real semantic search. One document is made close to the query.

In [5]:
docs = rng.normal(size=(1000, 64))
query = rng.normal(size=(1, 64))
docs[123] = query[0] + rng.normal(scale=0.1, size=64)   # plant a near-duplicate

scores = cosine_similarity(query, docs)[0]
best = top_k_indices(scores, 3)
print('top 3 indices:', best.tolist())
print('their scores :', scores[best].round(3).tolist())

top 3 indices: [123, 941, 349]
their scores : [0.997, 0.329, 0.317]


## 5. Softmax stability

The naive formula overflows for large logits; subtracting the max fixes it without changing the result.

In [6]:
logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(logits) / np.exp(logits).sum()
print('naive :', naive)
print('stable:', softmax(logits).round(4))

naive : [nan nan nan]
stable: [0.09   0.2447 0.6652]


## Takeaways

- Think in shapes; broadcasting replaces nested loops.
- Verify vectorized code against the simple loop before trusting it.
- Edge cases (overflow, zero variance) need explicit handling.